In [0]:
spark.table("workspace.logistics_project.safety_incidents").printSchema()

root
 |-- incident_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- incident_date: timestamp (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- at_fault_flag: boolean (nullable = true)
 |-- injury_flag: boolean (nullable = true)
 |-- vehicle_damage_cost: double (nullable = true)
 |-- cargo_damage_cost: double (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- preventable_flag: boolean (nullable = true)
 |-- description: string (nullable = true)



In [0]:
#Read Tables
import pyspark.sql.functions as F

safety = spark.table("workspace.logistics_project.safety_incidents")
drivers = spark.table("workspace.logistics_project.drivers")
trucks = spark.table("workspace.logistics_project.trucks")

In [0]:
#Join Tables
safety_analysis = (
    safety
    .join(drivers, "driver_id", "left")
    .join(trucks, "truck_id", "left")
)

In [0]:
#Calculate Safety KPIs
safety_metrics = (
    safety_analysis
    .groupBy(
        "driver_id",
        "first_name",
        "last_name",
        "truck_id",
        "unit_number"
    )
    .agg(
        F.count("incident_id").alias("total_incidents"),

        F.sum(
            F.when(F.col("at_fault_flag") == True, 1)
             .otherwise(0)
        ).alias("at_fault_incidents"),

        F.sum(
            F.when(F.col("preventable_flag") == True, 1)
             .otherwise(0)
        ).alias("preventable_incidents"),

        F.sum(
            F.when(F.col("injury_flag") == True, 1)
             .otherwise(0)
        ).alias("injury_incidents"),

        F.round(
            F.sum("vehicle_damage_cost"),2
        ).alias("vehicle_damage_cost"),

        F.round(
            F.sum("cargo_damage_cost"),2
        ).alias("cargo_damage_cost"),

        F.round(
            F.sum("claim_amount"),2
        ).alias("total_claim_amount")
    )
)

In [0]:
#Calculate Safety Rates
safety_metrics = (
    safety_metrics
    .withColumn(
        "at_fault_rate_pct",
        F.round(
            (F.col("at_fault_incidents") /
             F.col("total_incidents")) * 100,
            2
        )
    )
    .withColumn(
        "preventable_rate_pct",
        F.round(
            (F.col("preventable_incidents") /
             F.col("total_incidents")) * 100,
            2
        )
    )
)

In [0]:
#Create Safety Score
safety_metrics = safety_metrics.withColumn(
    "safety_score",
    F.round(
        100
        - (F.col("at_fault_rate_pct") * 0.4)
        - (F.col("preventable_rate_pct") * 0.3)
        - (F.col("injury_incidents") * 2),
        2
    )
)

In [0]:
#View Results
display(
    safety_metrics.orderBy(
        F.col("total_incidents").desc()
    )
)

driver_id,first_name,last_name,truck_id,unit_number,total_incidents,at_fault_incidents,preventable_incidents,injury_incidents,vehicle_damage_cost,cargo_damage_cost,total_claim_amount,at_fault_rate_pct,preventable_rate_pct,safety_score
DRV00121,Mary,Miller,TRK00019,1298,2,1,0,0,6982.35,4607.3,11589.65,50.0,0.0,80.0
DRV00038,William,Williams,TRK00082,3414,1,1,0,1,18303.46,0.0,18303.46,100.0,0.0,58.0
DRV00082,Michael,Anderson,TRK00103,3392,1,1,1,1,0.0,0.0,0.0,100.0,100.0,28.0
DRV00101,John,Davis,TRK00103,3392,1,1,1,0,0.0,0.0,0.0,100.0,100.0,30.0
DRV00065,Robert,Brown,TRK00015,2531,1,0,1,0,0.0,0.0,0.0,0.0,100.0,70.0
DRV00095,Linda,Thomas,TRK00048,4891,1,0,0,0,7869.76,0.0,7869.76,0.0,0.0,100.0
DRV00036,Michael,Thomas,TRK00019,1298,1,0,1,0,10368.65,3080.69,13449.34,0.0,100.0,70.0
DRV00133,Mary,Wilson,TRK00014,9624,1,0,0,0,14951.4,0.0,14951.4,0.0,0.0,100.0
DRV00122,James,Wilson,TRK00108,3273,1,0,0,0,12824.5,0.0,12824.5,0.0,0.0,100.0
DRV00001,Jennifer,Hernandez,TRK00031,5846,1,1,0,0,22667.73,0.0,22667.73,100.0,0.0,60.0


In [0]:
display(safety_metrics)

driver_id,first_name,last_name,truck_id,unit_number,total_incidents,at_fault_incidents,preventable_incidents,injury_incidents,vehicle_damage_cost,cargo_damage_cost,total_claim_amount,at_fault_rate_pct,preventable_rate_pct,safety_score
DRV00038,William,Williams,TRK00082,3414,1,1,0,1,18303.46,0.0,18303.46,100.0,0.0,58.0
DRV00082,Michael,Anderson,TRK00103,3392,1,1,1,1,0.0,0.0,0.0,100.0,100.0,28.0
DRV00101,John,Davis,TRK00103,3392,1,1,1,0,0.0,0.0,0.0,100.0,100.0,30.0
DRV00065,Robert,Brown,TRK00015,2531,1,0,1,0,0.0,0.0,0.0,0.0,100.0,70.0
DRV00095,Linda,Thomas,TRK00048,4891,1,0,0,0,7869.76,0.0,7869.76,0.0,0.0,100.0
DRV00036,Michael,Thomas,TRK00019,1298,1,0,1,0,10368.65,3080.69,13449.34,0.0,100.0,70.0
DRV00133,Mary,Wilson,TRK00014,9624,1,0,0,0,14951.4,0.0,14951.4,0.0,0.0,100.0
DRV00122,James,Wilson,TRK00108,3273,1,0,0,0,12824.5,0.0,12824.5,0.0,0.0,100.0
DRV00001,Jennifer,Hernandez,TRK00031,5846,1,1,0,0,22667.73,0.0,22667.73,100.0,0.0,60.0
DRV00103,Joseph,Hernandez,TRK00099,1461,1,0,0,0,16514.69,0.0,16514.69,0.0,0.0,100.0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|customer_analysis   |false      |
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
safety_metrics.write \
.format("delta") \
.saveAsTable(
    "workspace.logistics_gold.safety_analysis"
)

In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.safety_analysis"
)

gold_table.alias("target").merge(
    safety_metrics.alias("source"),
    """
    target.driver_id = source.driver_id
    AND
    target.truck_id = source.truck_id
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#verify
display(
    spark.table(
        "workspace.logistics_gold.safety_analysis"
    )
)

driver_id,first_name,last_name,truck_id,unit_number,total_incidents,at_fault_incidents,preventable_incidents,injury_incidents,vehicle_damage_cost,cargo_damage_cost,total_claim_amount,at_fault_rate_pct,preventable_rate_pct,safety_score
DRV00101,John,Davis,null,null,1,1,0,0,13414.05,0.0,13414.05,100.0,0.0,60.0
null,null,null,TRK00120,2821,1,0,1,0,0.0,0.0,0.0,0.0,100.0,70.0
DRV00038,William,Williams,TRK00082,3414,1,1,0,1,18303.46,0.0,18303.46,100.0,0.0,58.0
DRV00082,Michael,Anderson,TRK00103,3392,1,1,1,1,0.0,0.0,0.0,100.0,100.0,28.0
DRV00101,John,Davis,TRK00103,3392,1,1,1,0,0.0,0.0,0.0,100.0,100.0,30.0
DRV00065,Robert,Brown,TRK00015,2531,1,0,1,0,0.0,0.0,0.0,0.0,100.0,70.0
DRV00095,Linda,Thomas,TRK00048,4891,1,0,0,0,7869.76,0.0,7869.76,0.0,0.0,100.0
DRV00036,Michael,Thomas,TRK00019,1298,1,0,1,0,10368.65,3080.69,13449.34,0.0,100.0,70.0
DRV00133,Mary,Wilson,TRK00014,9624,1,0,0,0,14951.4,0.0,14951.4,0.0,0.0,100.0
DRV00122,James,Wilson,TRK00108,3273,1,0,0,0,12824.5,0.0,12824.5,0.0,0.0,100.0
